<a href="https://colab.research.google.com/github/KingTechnician/shared-truth/blob/main/tiu_calibrate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/KingTechnician/Truth_is_Universal.git

In [ ]:
import os
import glob
import sys
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Optional: Verify the token is active
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Import Truth is Universal libraries

In [ ]:
%cd /content/Truth_is_Universal


repo_path = '/content/Truth_is_Universal'
if repo_path not in sys.path:
    sys.path.append(repo_path)

try:
    import utils
    import calibration
    print("✅ Successfully imported Truth_is_Universal libraries.")

except ImportError as e:
    print(f"❌ Error importing libraries: {e}")
    print("Verify that the files (utils.py, generate_acts.py, probes.py) exist in " + repo_path)

Run their generate_acts script to generate activations at ALL layers

In [ ]:
%cd /content/Truth_is_Universal


!python generate_acts.py \
  --model_family Gemma \
  --model_size 7B \
  --model_type instruct \
  --layers -1 \
  --datasets cities neg_cities sp_en_trans neg_sp_en_trans \
  --output_dir acts \
  --device cuda:0


Use the calibration library to compute for the best truth layer within the model. Run a few times to get the most ranked layer - sometimes it will fluctuate.

We use a sample of the dataset and this seems enough for performant truth classification. Test with your own linear probes to verify.

In [ ]:
model = "Gemma"
params = "7B"
version = "instruct"

candidate_layers = calibration.infer_available_layers(model,params,version)

best_layer, layer_scores = calibration.calibrate_best_layer(
    model_family=model,
    model_size=params,
    model_type=version,
    layers=candidate_layers,
    datasets=["cities", "neg_cities", "sp_en_trans", "neg_sp_en_trans"],
    samples_per_dataset=50,
)
print("Best layer:", best_layer)
